# 🗣️ Azure AI Language — Lab AI-102

**Objectif**: Maîtriser l'analyse NLP avec Azure AI Language (anciennement Text Analytics).

## Compétences AI-102 couvertes
- Détection de langue avec scores de confiance
- Analyse de sentiment avec opinion mining
- Extraction de phrases clés
- Reconnaissance d'entités nommées (NER)
- Liaison d'entités vers une base de connaissance
- Détection et redaction de PII
- Résumé abstractif et extractif
- Exécuter plusieurs analyses en un seul appel API

In [ ]:
%pip install azure-ai-textanalytics python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

load_dotenv('../.env')

client = TextAnalyticsClient(
    endpoint=os.getenv('AZURE_LANGUAGE_ENDPOINT'),
    credential=AzureKeyCredential(os.getenv('AZURE_LANGUAGE_KEY'))
)

# Textes de test
SAMPLE_FR = """Microsoft Azure AI Foundry offre une suite complète de services d'intelligence artificielle. 
La plateforme permet aux développeurs de Paris et Lyon de créer des solutions innovantes rapidement. 
Le service de traitement du langage naturel est excellent, mais les prix sont parfois élevés."""

SAMPLE_EN = """John Smith, born on 12/15/1985, lives at 123 Main Street, New York. 
His email is john.smith@example.com and phone number is +1-555-0123.
He works at Microsoft Corporation as a Senior Software Engineer."""

print('✅ Client AI Language initialisé')

## 1. Détection de langue

In [ ]:
# AI-102: Detect Language retourne le code ISO 639-1, le nom et le score de confiance

texts = [
    {"id": "1", "text": "Bonjour, comment allez-vous aujourd'hui?"},
    {"id": "2", "text": "Hello, how are you today?"},
    {"id": "3", "text": "مرحبا، كيف حالك اليوم؟"},
    {"id": "4", "text": "你好，今天怎么样？"},
]

results = client.detect_language(documents=texts)

for doc in results:
    if not doc.is_error:
        lang = doc.primary_language
        print(f"ID {doc.id}: {lang.name} ({lang.iso6391_name}) — confiance: {lang.confidence_score:.2%}")
    else:
        print(f"ID {doc.id}: Erreur — {doc.error}")

## 2. Analyse de sentiment avec Opinion Mining

In [ ]:
# AI-102: Opinion Mining identifie les cibles (aspects) et les évaluations associées
# Ex: 'Le service est rapide mais l'interface est confuse' → service=positif, interface=négatif

results = client.analyze_sentiment(
    documents=[{"id": "1", "text": SAMPLE_FR, "language": "fr"}],
    show_opinion_mining=True  # Active l'opinion mining
)

for doc in results:
    if not doc.is_error:
        print(f"Sentiment global: {doc.sentiment.upper()}")
        print(f"Scores: Positif={doc.confidence_scores.positive:.0%} | Neutre={doc.confidence_scores.neutral:.0%} | Négatif={doc.confidence_scores.negative:.0%}")
        
        print("\nAnalyse par phrase:")
        for sentence in doc.sentences:
            print(f"  [{sentence.sentiment}] '{sentence.text[:60]}...'")
            
            # Opinion Mining
            for opinion in sentence.mined_opinions:
                target = opinion.target
                print(f"    → Cible: '{target.text}' ({target.sentiment})")
                for assessment in opinion.assessments:
                    neg = ' [NEGATION]' if assessment.is_negated else ''
                    print(f"      Évaluation: '{assessment.text}' ({assessment.sentiment}){neg}")

## 3. Extraction de phrases clés

In [ ]:
# AI-102: Key phrase extraction identifie les principaux sujets du texte

results = client.extract_key_phrases(
    documents=[{"id": "1", "text": SAMPLE_FR, "language": "fr"}]
)

for doc in results:
    if not doc.is_error:
        print(f"Phrases clés extraites ({len(doc.key_phrases)}):")
        for phrase in doc.key_phrases:
            print(f"  • {phrase}")

## 4. Reconnaissance d'entités nommées (NER)

In [ ]:
# AI-102: NER reconnaît: Person, Location, Organization, DateTime, Quantity, URL, Email, etc.

results = client.recognize_entities(
    documents=[{"id": "1", "text": SAMPLE_FR, "language": "fr"}]
)

for doc in results:
    if not doc.is_error:
        print(f"Entités reconnues ({len(doc.entities)}):")
        for entity in doc.entities:
            sub = f" → {entity.subcategory}" if entity.subcategory else ""
            print(f"  '{entity.text}' → {entity.category}{sub} ({entity.confidence_score:.0%})")

## 5. Liaison d'entités (Entity Linking)

In [ ]:
# AI-102: Entity Linking désambiguïse les entités et les lie à Wikipedia
# Utile pour distinguer 'Paris' (France) de 'Paris' (Texas)

results = client.recognize_linked_entities(
    documents=[{"id": "1", "text": SAMPLE_FR, "language": "fr"}]
)

for doc in results:
    if not doc.is_error:
        print(f"Entités liées ({len(doc.entities)}):")
        for entity in doc.entities:
            print(f"  {entity.name} ({entity.data_source})")
            print(f"    URL: {entity.url}")
            for match in entity.matches:
                print(f"    Mention: '{match.text}' (confiance: {match.confidence_score:.0%})")

## 6. Détection et redaction de PII

In [ ]:
# AI-102: PII detection identifie: Email, Phone, SSN, CreditCard, Name, Address, etc.
# La redaction remplace les PII par des astérisques

results = client.recognize_pii_entities(
    documents=[{"id": "1", "text": SAMPLE_EN, "language": "en"}]
)

for doc in results:
    if not doc.is_error:
        print("Entités PII détectées:")
        for entity in doc.entities:
            print(f"  '{entity.text}' → {entity.category} ({entity.confidence_score:.0%})")
        
        print(f"\nTexte original:\n{SAMPLE_EN}")
        print(f"\nTexte anonymisé:\n{doc.redacted_text}")

## 7. Résumé abstractif et extractif

In [ ]:
# AI-102: Summarization
# - Extractif: sélectionne les phrases clés du texte original
# - Abstractif: génère un nouveau résumé (utilise un modèle de langage)

from azure.ai.textanalytics import AbstractiveSummaryAction, ExtractiveSummaryAction

long_text = """Azure AI Foundry est une plateforme unifiée pour le développement d'applications d'IA.
Elle intègre des outils comme Azure OpenAI Service, Azure AI Language, Azure AI Vision et bien d'autres.
Les développeurs peuvent créer, former et déployer des modèles d'IA directement depuis l'interface.
La plateforme supporte le RAG (Retrieval Augmented Generation) pour améliorer la précision des réponses.
Azure AI Search permet d'indexer des documents et d'effectuer des recherches sémantiques avancées.
Les entreprises utilisent ces outils pour automatiser l'analyse documentaire, le service client et la détection de fraude.
La certification AI-102 valide les compétences pour concevoir et implémenter des solutions Azure AI."""

poller = client.begin_analyze_actions(
    documents=[{"id": "1", "text": long_text, "language": "fr"}],
    actions=[
        AbstractiveSummaryAction(sentence_count=2),
        ExtractiveSummaryAction(sentence_count=2),
    ]
)

results = poller.result()
for page in results:
    for i, action_result in enumerate(page):
        if not action_result.is_error:
            if i == 0:  # Abstractif
                print("Résumé ABSTRACTIF (nouveau texte généré):")
                for summary in action_result.summaries:
                    print(f"  {summary.text}")
            else:  # Extractif
                print("\nRésumé EXTRACTIF (phrases sélectionnées):")
                for sentence in action_result.sentences:
                    print(f"  [{sentence.rank_score:.2f}] {sentence.text}")

## 8. Analyse multi-actions en un seul appel (efficacité API)

In [ ]:
# AI-102: begin_analyze_actions permet d'exécuter plusieurs analyses en parallèle
# Optimise les coûts et la latence

from azure.ai.textanalytics import (
    AnalyzeSentimentAction,
    ExtractKeyPhrasesAction,
    RecognizeEntitiesAction,
    RecognizePiiEntitiesAction,
)

poller = client.begin_analyze_actions(
    documents=[{"id": "1", "text": SAMPLE_FR, "language": "fr"}],
    actions=[
        AnalyzeSentimentAction(show_opinion_mining=True),
        ExtractKeyPhrasesAction(),
        RecognizeEntitiesAction(),
    ]
)

results = poller.result()
action_names = ["Sentiment", "Phrases clés", "Entités NER"]

for page in results:
    for i, action_result in enumerate(page):
        print(f"\n=== {action_names[i]} ===")
        if not action_result.is_error:
            if i == 0:
                print(f"Sentiment: {action_result.sentiment}")
            elif i == 1:
                print(f"Phrases: {list(action_result.key_phrases)[:5]}")
            elif i == 2:
                print(f"Entités: {[(e.text, e.category) for e in action_result.entities][:5]}")